# QRE2 Dense Carleman Lift Handoff

This notebook is the visual handoff for the smallest private dense Carleman validation in this repository. It uses the validated `QRE2` shifted-D2Q9 polynomial operator, then builds only the `N_C=1/2` lifted recurrence needed for tiny classical checks.

Pedagogical sequence: paper anchor -> compact equation -> operator or matrix object -> observable plot -> falsifiable check.

Corpus anchors: `QRE2` (`sec:discrete_carleman`, `eq:Ckl`, `eq:carl_evol_d`, `eq:LBE_recurrence`), comparator/warnings `CAR7`, `IO1`, `CAR15`, `CAR9`, `CAR19`, and encoding/readout gates `IO4`, `IO5`, `IO7`, `IO8`.

## Lifted Equation

The validated shifted update is a quadratic map:

$$
g_{t+1}=S\left[(I+F_1)g_t + F_2(g_t\otimes g_t)\right].
$$

For `N_C=2`, the dense Carleman state and propagator are:

$$
y_t=[g_t, g_t\otimes g_t],\qquad y_{t+1}=P y_t,\qquad P=S_C C.
$$

The observable validation target is only the first block. This notebook does not claim a loading oracle, block encoding, QLSA solve, circuit, readout protocol, or resource estimate.

In [ ]:
# Corpus anchors: QRE2 (`sec:discrete_carleman`, `eq:Ckl`, `eq:carl_evol_d`, `eq:LBE_recurrence`), CAR7, IO1, CAR15, CAR9, CAR19, IO4, IO5, IO7, IO8.
from pathlib import Path
import sys
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pq_cfd import _qre2_carleman as carleman
from pq_cfd import _qre2_shifted as qre2
sns.set_theme(context='notebook', style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 130})

def small_shifted_state(spatial_shape):
    nx, ny = spatial_shape
    x = np.arange(nx, dtype=float)[:, None]
    y = np.arange(ny, dtype=float)[None, :]
    delta_rho = 0.01 * np.cos(2.0 * np.pi * (x + y) / (nx + ny))
    velocity = np.zeros((2, nx, ny))
    velocity[0] = 0.02 * np.sin(2.0 * np.pi * (x + 1.0) / (nx + 1.0))
    velocity[1] = -0.015 * np.cos(2.0 * np.pi * (y + 1.0) / (ny + 1.0))
    return qre2.shifted_equilibrium_from_moments(delta_rho, velocity)

def add_block_boundaries(ax, block_sizes, color='white'):
    offsets = np.cumsum(block_sizes)
    for offset in offsets[:-1]:
        ax.axhline(offset - 0.5, color=color, linewidth=1.0)
        ax.axvline(offset - 0.5, color=color, linewidth=1.0)

def log_abs(matrix):
    return np.log10(np.abs(matrix) + 1e-18)
print(f'project_root={project_root}')
print('dense Carleman notebook boundary: classical validation only')

: 

## Operator Object And Lifted-State Size

Use a tiny periodic `1x3` grid so dense matrices remain audit objects. The base dimension is `d = N Q`, and the `N_C=2` lifted vector has dimension `d + d^2`.

This dimension plot is not a resource estimate. It is a warning object: even the smallest useful lift grows quickly enough that `IO1`, `CAR15`, `IO4`, `IO5`, `IO7`, and `IO8` must be resolved before quantum/resource claims.

In [ ]:
# Corpus anchors: QRE2 (`eq:carl_evol_d`, `eq:LBE_recurrence`), IO1, CAR15, IO4, IO5, IO7, IO8. Lifted dimension and guard visualization.
spatial_shape = (1, 3)
tau_bar = 0.83
g0 = qre2.flatten_site_major(small_shifted_state(spatial_shape))
f1, f2 = qre2.build_shifted_collision_terms(tau_bar, spatial_shape)
streaming = qre2.build_streaming_matrix(spatial_shape)
base_dimension = g0.size
block_sizes = carleman.carleman_block_dimensions(base_dimension, order=2)
total_dimension = carleman.carleman_dimension(base_dimension, order=2)
candidate_shapes = [(1, 1), (1, 2), (1, 3), (2, 2), (2, 3)]
candidate_base_dims = np.array([np.prod(shape) * 9 for shape in candidate_shapes], dtype=int)
candidate_lift_dims = np.array([carleman.carleman_dimension(int(dim), order=2) for dim in candidate_base_dims], dtype=int)
_fig, _axes = plt.subplots(1, 2, figsize=(12, 4))
_axes[0].bar(['g', 'g tensor g'], block_sizes, color=sns.color_palette('colorblind', 2))
_axes[0].set_title(f'Lifted state blocks for grid {spatial_shape}')
_axes[0].set_ylabel('entries')
for _index, value in enumerate(block_sizes):
    _axes[0].text(_index, value, str(value), ha='center', va='bottom')
_axes[1].bar([f'{shape[0]}x{shape[1]}' for shape in candidate_shapes], candidate_lift_dims, color=sns.color_palette('crest', len(candidate_shapes)))
_axes[1].axhline(carleman.DEFAULT_MAX_DIMENSION, color='0.35', linestyle='--', label='default dense guard')
_axes[1].set_title('Dense N_C=2 dimension guard')
_axes[1].set_xlabel('tiny grid')
_axes[1].set_ylabel('d + d^2')
_axes[1].legend(fontsize=8)
_axes[1].tick_params(axis='x', rotation=25)
_fig.suptitle('QRE2 Carleman lift size: validation object, not a resource model', y=1.03)
_fig.tight_layout()
print(f'base dimension={base_dimension}')
print(f'Carleman blocks={block_sizes}')
print(f'total lifted dimension={total_dimension}')
assert total_dimension <= carleman.DEFAULT_MAX_DIMENSION

## Dense Block Matrices

The dense validation objects are:

$$
C = \begin{bmatrix} C_{11} & C_{12} \\ 0 & C_{22} \end{bmatrix},\qquad
S_C=\mathrm{diag}(S,S\otimes S),\qquad P=S_C C.
$$

The block boundaries are drawn on the heatmaps. These are tiny classical matrices used to check algebraic behavior, not matrix-access oracles.

In [ ]:
# Corpus anchors: QRE2 (`eq:Ckl`, `eq:carl_evol_d`), CAR7, IO1, CAR15. Dense block-matrix visualization.
collision = carleman.build_carleman_collision(f1, f2, order=2)
lifted_streaming = carleman.build_carleman_streaming(streaming, order=2)
propagator = carleman.build_carleman_propagator(streaming, f1, f2, order=2)
_fig, _axes = plt.subplots(2, 3, figsize=(13, 8))
matrices = [(collision, 'C: lifted collision'), (lifted_streaming, 'S_C: lifted streaming'), (propagator, 'P = S_C C')]
for _ax, (matrix, _title) in zip(_axes[0], matrices):
    image = _ax.imshow(log_abs(matrix), cmap='mako', aspect='auto')
    add_block_boundaries(_ax, block_sizes)
    _ax.set_title(_title)
    _ax.set_xlabel('column')
    _ax.set_ylabel('row')
    _fig.colorbar(image, ax=_ax, fraction=0.046, pad=0.04, label='log10(|entry|+1e-18)')
for _ax, (matrix, _title) in zip(_axes[1], matrices):
    _ax.spy(np.abs(matrix) > 1e-15, markersize=0.25)
    add_block_boundaries(_ax, block_sizes, color='red')
    _ax.set_title(f'sparsity: {_title}')
    _ax.set_xlabel('column')
    _ax.set_ylabel('row')
_fig.suptitle('Dense QRE2 N_C=2 Carleman matrices with visible block structure', y=1.02)
_fig.tight_layout()
for name, matrix in (('C', collision), ('S_C', lifted_streaming), ('P', propagator)):
    nonzero = int(np.count_nonzero(np.abs(matrix) > 1e-15))
    print(f'{name}: shape={matrix.shape}, nonzero={nonzero}, density={nonzero / matrix.size:.3e}')
assert propagator.shape == (total_dimension, total_dimension)

## Falsifiable One-Step Check

The first lifted block must match the direct shifted polynomial update for one step:

$$
\mathrm{first\_block}(P [g,g\otimes g]) = S\left[(I+F_1)g+F_2(g\otimes g)\right].
$$

This verifies the recurrence assembly only. It says nothing about loading, readout, normalization, success probability, or circuit cost.

In [ ]:
# Corpus anchors: QRE2 (`eq:LBE_recurrence`, `eq:LBE_col_shift_matrix`). One-step first-block check.
lifted_state = carleman.lift_shifted_state(g0, order=2)
lifted_next = propagator @ lifted_state
first_block = carleman.extract_first_block(lifted_next, g0.size)
direct_next = qre2.step_shifted_matrix(g0, streaming, f1, f2)
one_step_residual = np.linalg.norm(first_block - direct_next)
_fig, _axes = plt.subplots(1, 2, figsize=(12, 4))
_axes[0].plot(first_block, marker='o', markersize=3, linewidth=1, label='Carleman first block')
_axes[0].plot(direct_next, linestyle='--', linewidth=1, label='direct polynomial update')
_axes[0].set_title('First-block values after one step')
_axes[0].set_xlabel('base vector index')
_axes[0].legend(fontsize=8)
_axes[1].stem(np.abs(first_block - direct_next), basefmt=' ')
_axes[1].set_yscale('log')
_axes[1].set_title('Absolute first-block residual')
_axes[1].set_xlabel('base vector index')
_axes[1].set_ylabel('abs residual')
_fig.suptitle('QRE2 dense Carleman one-step validation', y=1.03)
_fig.tight_layout()
print(f'one-step first-block residual={one_step_residual:.3e}')
assert one_step_residual < 1e-12

## N_C=1 Versus N_C=2 Trajectory Check

The `N_C=1` lift keeps only the linear-control part. The `N_C=2` lift keeps the quadratic block needed for the first-block update to track the direct shifted polynomial map better over a short tiny-grid run.

The pass condition is qualitative but falsifiable here: the final `N_C=2` relative first-block error must be smaller than the final `N_C=1` error.

In [ ]:
# Corpus anchors: QRE2 (`eq:carl_evol_d`, `eq:LBE_recurrence`), CAR7. N_C=1 vs N_C=2 first-block error trajectory.
def first_block_error_trajectory(order, steps=6):
    direct = g0.copy()
    lifted = carleman.lift_shifted_state(g0, order=order)
    P = carleman.build_carleman_propagator(streaming, f1, f2, order=order)
    errors = [0.0]
    for _ in range(steps):
        direct = qre2.step_shifted_matrix(direct, streaming, f1, f2)
        lifted = P @ lifted
        approximate = carleman.extract_first_block(lifted, g0.size)
        errors.append(float(np.linalg.norm(approximate - direct) / np.linalg.norm(direct)))
    return np.array(errors)
nc1_errors = first_block_error_trajectory(order=1, steps=6)
nc2_errors = first_block_error_trajectory(order=2, steps=6)
_fig, _ax = plt.subplots(figsize=(7, 4))
_ax.semilogy(range(nc1_errors.size), np.maximum(nc1_errors, 1e-18), marker='o', label='N_C=1')
_ax.semilogy(range(nc2_errors.size), np.maximum(nc2_errors, 1e-18), marker='s', label='N_C=2')
_ax.set_title('First-block error against direct QRE2 polynomial trajectory')
_ax.set_xlabel('step')
_ax.set_ylabel('relative first-block error')
_ax.legend(fontsize=8)
_fig.tight_layout()
print(f'N_C=1 final relative first-block error={nc1_errors[-1]:.3e}')
print(f'N_C=2 final relative first-block error={nc2_errors[-1]:.3e}')
assert np.isfinite(nc2_errors[-1])
assert nc2_errors[-1] < nc1_errors[-1]

## Explicit Non-Claims

The visual objects above are useful precisely because they keep the boundary visible. The dense lift is a classical validation artifact. The warnings below must be resolved in a route note before quantum/resource code is added.

In [ ]:
# Corpus anchors: QRE2, CAR7, IO1, CAR15, CAR9, CAR19, IO4, IO5, IO7, IO8. Explicit route-boundary warning tiles.
warnings = [('No loading oracle', 'IO1, CAR15'), ('No block encoding', 'IO4, IO5'), ('No QLSA solve', 'QRE2 boundary'), ('No circuit claim', 'CAR9, CAR19'), ('No readout model', 'IO7, IO8'), ('No resource estimate', 'QRE2, CAR7')]
_fig, _ax = plt.subplots(figsize=(12, 3.5))
_ax.set_xlim(0, 3)
_ax.set_ylim(0, 2)
_ax.axis('off')
for _index, (_title, anchor) in enumerate(warnings):
    x = _index % 3
    y = 1 - _index // 3
    _ax.text(x + 0.5, y + 0.5, f'{_title}\n{anchor}', ha='center', va='center', fontsize=11, bbox={'boxstyle': 'round,pad=0.45', 'fc': '#fff7ec', 'ec': '#d95f02', 'lw': 1.5})
_ax.set_title('Route-boundary warnings before quantum/resource implementation')
_fig.tight_layout()
print('Dense Carleman validation stops before loading, encoding, readout, circuit, and resource claims.')

## Research Boundary

This notebook stops at the dense recurrence because `IO1` and `CAR15` make data loading and matrix-oracle assumptions first-class blockers, while `CAR9` and `CAR19` warn against treating linearized representations as automatic speedups.

The next route decision must still specify encoding, loading/reloading, readout, normalization, success probability, and resource quantities before any quantum package is added.

Verification command:

```powershell
uv run pytest tests/test_qre2_carleman.py -q
uv run --extra notebook jupyter nbconvert --to notebook --execute --output-dir .cache/notebooks notebooks/qre2_carleman_lift_handoff.ipynb
uv run --extra notebook jupyter lab notebooks/qre2_carleman_lift_handoff.ipynb
```